In [1]:
import pandas as pd

In [2]:
#load the cleaned version
data = pd.read_csv("clean_observations.csv")

In [3]:
print(data.head())
print(data.shape)
print(data.columns)

         lat        lon  taxon_id                   name
0 -18.286728  143.48125     31529  Lophognathus gilberti
1 -13.099798  130.78365     31529  Lophognathus gilberti
2 -13.965274  131.69514     31529  Lophognathus gilberti
3 -12.853950  132.80050     31529  Lophognathus gilberti
4 -12.196790  134.27933     31529  Lophognathus gilberti
(252464, 4)
Index(['lat', 'lon', 'taxon_id', 'name'], dtype='object')


In [4]:
#step 1 pseudo absence
#define a function for all 15 species
def species_loop(taxon_id, ratio=1):
    presence = data[data["taxon_id"] == taxon_id].drop_duplicates(subset=["lat", "lon"]).copy()
    presence["label"] = 1
#form background data
    background = data[data["taxon_id"] != taxon_id].copy()
#remove locations matching known presence,left=background
    background = background.merge(presence[["lat", "lon"]].drop_duplicates(),on=["lat", "lon"],how="left",indicator=True)
#keep different ones
    background = background[background["_merge"] == "left_only"].drop(columns="_merge")
#remove duplicate background locations
    background = background.drop_duplicates(subset=["lat", "lon"])
#generate pseudo absence
    pseudo_absence = background.sample(n=len(presence)*ratio,random_state=0).copy()
    pseudo_absence["label"] = 0
#set target species ID and name
    pseudo_absence["taxon_id"] = taxon_id
    pseudo_absence["name"] = presence["name"].iloc[0]
#combine
    training_data = pd.concat([presence, pseudo_absence],ignore_index=True)
#reform
    training_data = training_data.sample(frac=1,random_state=0).reset_index(drop=True)
    return training_data

In [5]:
#generate 1:1
species_list = pd.read_csv("species_list.csv")
taxon_ids = species_list["taxon_id"]
#generate the training data
pa_data = []
for taxon_id in taxon_ids:
    training_data = species_loop(taxon_id,ratio=1)
    pa_data.append(training_data)
#combine all 15 species into one DataFrame
all_data = pd.concat(pa_data, ignore_index=True)
#save
all_data.to_csv("pseudoabsence_1:1.csv", index=False)

In [6]:
#generate 1:5
#generate the training data
pa_data = []
for taxon_id in taxon_ids:
    training_data = species_loop(taxon_id,ratio=5)
    pa_data.append(training_data)
#combine all 15 species into one DataFrame
all_data = pd.concat(pa_data, ignore_index=True)
#save
all_data.to_csv("pseudoabsence_1:5.csv", index=False)

In [7]:
import numpy as np

In [8]:
#define a function that can be applied to both ratio
#step 2 feature
def fun_data(all_data):
    all_data = all_data.copy()
#convert degrees to radians
    lat_rad = np.radians(all_data["lat"])
    lon_rad = np.radians(all_data["lon"])
#convert to 3D
    all_data["x"] = np.cos(lat_rad) * np.cos(lon_rad)
    all_data["y"] = np.cos(lat_rad) * np.sin(lon_rad)
    all_data["z"] = np.sin(lat_rad)
#step 3 spatial cv
#define grid size
    grid_size = 10
#create geographic blocks
    all_data["lat_block"] = ((all_data["lat"] + 90) // grid_size).astype(int)
    all_data["lon_block"] = ((all_data["lon"] + 180) // grid_size).astype(int)
#combine blocks
    all_data["spatial_block"] = (all_data["lat_block"].astype(str) + "_" +all_data["lon_block"].astype(str))
    return all_data

In [9]:
#load both datasets
data1 = pd.read_csv("pseudoabsence_1:1.csv")
data2 = pd.read_csv("pseudoabsence_1:5.csv")
data1 = fun_data(data1)
data2 = fun_data(data2)
#combine
combined_data = pd.concat([data1,data2], ignore_index=True)

In [10]:
from sklearn.model_selection import GroupKFold
#create five folds
gkf = GroupKFold(n_splits=5)
#create a column to store fold IDs
combined_data["fold_id"] = -1
#split all species
for fold, (train_idx, val_idx) in enumerate(gkf.split(combined_data,groups=combined_data["spatial_block"])):
    combined_data.iloc[val_idx,combined_data.columns.get_loc("fold_id")] = fold
#build a map
fold_mapping = combined_data.drop_duplicates("spatial_block").set_index("spatial_block")["fold_id"]
#apply to both datasets
data1["fold_id"] = data1["spatial_block"].map(fold_mapping)
data2["fold_id"] = data2["spatial_block"].map(fold_mapping)
#save
data1.to_csv("spatial_folds_1:1.csv", index=False)
data2.to_csv("spatial_folds_1:5.csv", index=False)

In [11]:
#random 5-fold cv
from sklearn.model_selection import KFold
def random_folds(rdata):
    rdata = rdata.copy()
    rdata["random_fold_id"] = -1
    kf = KFold(n_splits=5,shuffle=True,random_state=0)
    for fold, (train_idx, val_idx) in enumerate(kf.split(rdata)):
        rdata.iloc[val_idx, rdata.columns.get_loc("random_fold_id")] = fold
    return rdata
#1:1
random1 = random_folds(data1)
random1.to_csv("randomfolds_1:1.csv", index=False)
#1:5
random2 = random_folds(data2)
random2.to_csv("randomfolds_1:5.csv", index=False)